#  Preparação de Dados - Trabalho Estudantil
## Extração e Tratamento das Variáveis Q007 e Q008

**Notebook 2/7** - Série: Trabalho Estudantil e Desempenho no ENEM

---

##  Objetivos

1. Carregar o dataset original do ENEM 2023
2. Extrair variáveis relacionadas ao trabalho estudantil (Q007, Q008)
3. Integrar com variáveis já processadas (notas, socioeconômicas)
4. Realizar limpeza e tratamento de dados
5. Criar variáveis derivadas para análise
6. Salvar dataset processado para análises posteriores

---

## 1⃣ Setup Inicial

In [35]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuração
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

print(" Bibliotecas carregadas")

 Bibliotecas carregadas


In [36]:
# Definir caminhos
PROJECT_ROOT = Path('/home/interas/faculdade/ciencia-dados/enem-data-exploration')
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'interim' / 'unzipped_2023' / 'DADOS'
PROCESSED_DIR = DATA_DIR / 'processed'

# Arquivos
RAW_FILE = RAW_DIR / 'MICRODADOS_ENEM_2023.csv'
EXISTING_FILE = PROCESSED_DIR / 'enem_2023.parquet'  # Dataset já processado
OUTPUT_FILE = PROCESSED_DIR / 'enem_2023_trabalho_estudantil.parquet'

print(f" Dataset original: {RAW_FILE.exists()}")
print(f" Dataset processado existente: {EXISTING_FILE.exists()}")

 Dataset original: True
 Dataset processado existente: False


---

## 2⃣ Carregar Dataset Existente

Primeiro, vamos carregar o dataset já processado que contém as notas e variáveis socioeconômicas básicas:

In [37]:
%%time
# ESTRATÉGIA: Carregar dataset completo e aplicar amostragem de 10.000 registros
SAMPLE_SIZE = 10000

# Verificar se dataset processado existe
if EXISTING_FILE.exists():
    print("Carregando dataset processado existente...")
    df_base = pd.read_parquet(EXISTING_FILE)
    print(f"Dataset processado carregado: {len(df_base):,} registros")
else:
    print("Dataset processado não encontrado.")
    print("Carregando do arquivo original (apenas colunas essenciais)...")
    
    # Carregar apenas colunas essenciais do arquivo original
    colunas_base = [
        'NU_INSCRICAO', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 
        'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'Q006', 'TP_ESCOLA', 
        'TP_SEXO', 'SG_UF_PROVA'
    ]
    df_base = pd.read_csv(RAW_FILE, sep=';', encoding='latin1', 
                          usecols=colunas_base, low_memory=False)
    
    print(f"Dados brutos carregados: {len(df_base):,} registros")
    
    # Criar variável de nota média
    notas_cols = ['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']
    df_base['NOTA_MEDIA_5'] = df_base[notas_cols].mean(axis=1)

# AMOSTRAGEM: Aplicar amostragem aleatória de 10.000 registros
if len(df_base) > SAMPLE_SIZE:
    print(f"\n🎲 Aplicando amostragem aleatória simples:")
    print(f"   População original: {len(df_base):,} registros")
    print(f"   Tamanho da amostra: {SAMPLE_SIZE:,} registros")
    print(f"   Percentual: {(SAMPLE_SIZE/len(df_base))*100:.2f}%")
    
    df_base = df_base.sample(n=SAMPLE_SIZE, random_state=42)
    
    print(f"Amostra selecionada: {len(df_base):,} registros")
else:
    print(f"\nDataset tem {len(df_base):,} registros (menor que {SAMPLE_SIZE:,})")
    print(f"   Usando todos os registros disponíveis")

print(f"\nDataset base final:")
print(f" Linhas: {len(df_base):,}")
print(f" Colunas: {len(df_base.columns)}")
print(f"\nColunas disponíveis:")
print(df_base.columns.tolist())

Dataset processado não encontrado.
Carregando do arquivo original (apenas colunas essenciais)...
Dados brutos carregados: 3,933,955 registros
Dados brutos carregados: 3,933,955 registros

🎲 Aplicando amostragem aleatória simples:
   População original: 3,933,955 registros
   Tamanho da amostra: 10,000 registros
   Percentual: 0.25%

🎲 Aplicando amostragem aleatória simples:
   População original: 3,933,955 registros
   Tamanho da amostra: 10,000 registros
   Percentual: 0.25%
Amostra selecionada: 10,000 registros

Dataset base final:
 Linhas: 10,000
 Colunas: 11

Colunas disponíveis:
['NU_INSCRICAO', 'TP_SEXO', 'TP_ESCOLA', 'SG_UF_PROVA', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'Q006', 'NOTA_MEDIA_5']
CPU times: user 19.5 s, sys: 16.9 s, total: 36.4 s
Wall time: 51.7 s
Amostra selecionada: 10,000 registros

Dataset base final:
 Linhas: 10,000
 Colunas: 11

Colunas disponíveis:
['NU_INSCRICAO', 'TP_SEXO', 'TP_ESCOLA', 'SG_UF_PROVA', 'NU_NOTA_CN', 'NU_N

In [38]:
# Primeiras linhas
df_base.head()

,NU_INSCRICAO,TP_SEXO,TP_ESCOLA,SG_UF_PROVA,NU_NOTA_CN,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,NU_NOTA_REDACAO,Q006,NOTA_MEDIA_5
553635,210059583535,F,2,RJ,408.9,526.9,417.9,416.5,720.0,B,498.04
3349365,210060187802,F,1,SP,499.4,535.6,549.2,570.7,560.0,D,542.98
2297393,210059061595,M,1,MA,425.2,391.7,446.0,503.5,460.0,B,445.28
1082491,210059417471,M,1,PE,620.8,584.4,493.0,412.7,820.0,C,586.18
1894282,210058647330,F,2,PR,445.1,458.0,457.2,491.6,400.0,B,450.38


In [39]:
# Informações sobre o dataset
df_base.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 553635 to 3134559
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   NU_INSCRICAO     10000 non-null  int64  
 1   TP_SEXO          10000 non-null  object 
 2   TP_ESCOLA        10000 non-null  int64  
 3   SG_UF_PROVA      10000 non-null  object 
 4   NU_NOTA_CN       6754 non-null   float64
 5   NU_NOTA_CH       7107 non-null   float64
 6   NU_NOTA_LC       7107 non-null   float64
 7   NU_NOTA_MT       6754 non-null   float64
 8   NU_NOTA_REDACAO  7107 non-null   float64
 9   Q006             10000 non-null  object 
 10  NOTA_MEDIA_5     7148 non-null   float64
dtypes: float64(6), int64(2), object(3)
memory usage: 937.5+ KB


---

## 3⃣ Extrair Variáveis de Trabalho do Dataset Original

Agora vamos ler apenas as colunas Q007 e Q008 do dataset original:

In [40]:
%%time
# Colunas a serem extraídas
colunas_trabalho = ['NU_INSCRICAO', 'Q007', 'Q008']

print("Lendo variáveis de trabalho do arquivo original...")
# Ler apenas as colunas necessárias
df_trabalho_completo = pd.read_csv(
    RAW_FILE,
    sep=';',
    encoding='latin1',
    usecols=colunas_trabalho,
    low_memory=False
)

# Filtrar apenas os registros que estão na amostra
inscricoes_amostra = df_base['NU_INSCRICAO'].unique()
df_trabalho = df_trabalho_completo[df_trabalho_completo['NU_INSCRICAO'].isin(inscricoes_amostra)].copy()

print(f"\n Variáveis de trabalho extraídas:")
print(f"  Linhas (população completa): {len(df_trabalho_completo):,}")
print(f"  Linhas (amostra filtrada): {len(df_trabalho):,}")
print(f"  Colunas: {list(df_trabalho.columns)}")

Lendo variáveis de trabalho do arquivo original...

 Variáveis de trabalho extraídas:
  Linhas (população completa): 3,933,955
  Linhas (amostra filtrada): 10,000
  Colunas: ['NU_INSCRICAO', 'Q007', 'Q008']
CPU times: user 14.7 s, sys: 12.7 s, total: 27.4 s
Wall time: 36.7 s

 Variáveis de trabalho extraídas:
  Linhas (população completa): 3,933,955
  Linhas (amostra filtrada): 10,000
  Colunas: ['NU_INSCRICAO', 'Q007', 'Q008']
CPU times: user 14.7 s, sys: 12.7 s, total: 27.4 s
Wall time: 36.7 s


In [41]:
# Primeiras linhas
df_trabalho.head(10)

,NU_INSCRICAO,Q007,Q008
198,210059982331,A,B
1377,210058581151,A,C
1713,210059010955,A,B
1973,210059568961,A,B
3752,210061104086,A,B
5361,210060388808,A,B
5661,210058593914,A,C
5719,210059301400,A,B
5840,210059910727,A,C
6169,210058587875,A,B


In [42]:
# Verificar valores únicos
print(" Q007 - Situação de trabalho:")
print(df_trabalho['Q007'].value_counts(dropna=False).sort_index())

print("\n Q008 - Carga horária:")
print(df_trabalho['Q008'].value_counts(dropna=False).sort_index())

 Q007 - Situação de trabalho:
Q007
A    9248
B     417
C     103
D     232
Name: count, dtype: int64

 Q008 - Carga horária:
Q008
A      97
B    6700
C    2201
D     662
E     340
Name: count, dtype: int64


---

## 4⃣ Integrar Dados

Vamos juntar as variáveis de trabalho com o dataset base:

In [43]:
# Verificar se NU_INSCRICAO existe no dataset base
if 'NU_INSCRICAO' in df_base.columns:
    print(" NU_INSCRICAO encontrado no dataset base")
    coluna_join = 'NU_INSCRICAO'
else:
    # Se não existir, precisamos ler do arquivo original
    print(" NU_INSCRICAO não encontrado. Lendo índice do arquivo original...")
    df_inscricao = pd.read_csv(
        RAW_FILE,
        sep=';',
        encoding='latin1',
        usecols=['NU_INSCRICAO'],
        low_memory=False
    )
    df_base['NU_INSCRICAO'] = df_inscricao['NU_INSCRICAO']
    coluna_join = 'NU_INSCRICAO'
    print(" NU_INSCRICAO adicionado ao dataset base")

 NU_INSCRICAO encontrado no dataset base


In [44]:
# Realizar merge
df = df_base.merge(df_trabalho, on=coluna_join, how='left')

print(f"\n Merge concluído:")
print(f"  Linhas: {len(df):,}")
print(f"  Colunas: {len(df.columns)}")
print(f"\n Novas colunas adicionadas: Q007, Q008")


 Merge concluído:
  Linhas: 10,000
  Colunas: 13

 Novas colunas adicionadas: Q007, Q008


In [45]:
# Verificar dados ausentes
print(" Valores ausentes:")
print(df[['Q007', 'Q008']].isnull().sum())
print(f"\nTaxa de preenchimento Q007: {(1 - df['Q007'].isnull().mean()) * 100:.2f}%")
print(f"Taxa de preenchimento Q008: {(1 - df['Q008'].isnull().mean()) * 100:.2f}%")

 Valores ausentes:
Q007    0
Q008    0
dtype: int64

Taxa de preenchimento Q007: 100.00%
Taxa de preenchimento Q008: 100.00%


---

## 5⃣ Tratamento de Dados

### 5.1 Análise de Valores Ausentes

In [46]:
# Verificar se valores ausentes são aleatórios ou sistemáticos
print(" Análise de valores ausentes em Q007:")
print(f"\nTotal de ausentes: {df['Q007'].isnull().sum():,} ({df['Q007'].isnull().mean()*100:.2f}%)")

# Comparar com outras variáveis
print("\n Correlação de ausência com outras variáveis:")
if 'Q006' in df.columns:  # Renda
    print(f"  Q006 (Renda) também ausente: {(df['Q007'].isnull() & df['Q006'].isnull()).sum():,}")
if 'TP_ESCOLA' in df.columns:
    print(f"  Distribuição por escola:")
    print(df.groupby('TP_ESCOLA')['Q007'].apply(lambda x: x.isnull().mean() * 100))

 Análise de valores ausentes em Q007:

Total de ausentes: 0 (0.00%)

 Correlação de ausência com outras variáveis:
  Q006 (Renda) também ausente: 0
  Distribuição por escola:
TP_ESCOLA
1    0.0
2    0.0
3    0.0
Name: Q007, dtype: float64
TP_ESCOLA
1    0.0
2    0.0
3    0.0
Name: Q007, dtype: float64


### 5.2 Criar Labels Descritivos

In [47]:
# Mapeamento Q007 - Situação de trabalho
Q007_LABELS = {
    'A': 'Não trabalho',
    'B': 'Trabalho eventualmente',
    'C': 'Trabalho meio período',
    'D': 'Trabalho período integral'
}

# Mapeamento Q008 - Carga horária
Q008_LABELS = {
    'A': 'Nenhuma',
    'B': 'Até 10 horas',
    'C': '11 a 20 horas',
    'D': '21 a 30 horas',
    'E': '31 a 40 horas',
    'F': 'Mais de 40 horas'
}

# Aplicar mapeamentos
df['Q007_label'] = df['Q007'].map(Q007_LABELS)
df['Q008_label'] = df['Q008'].map(Q008_LABELS)

print(" Labels descritivos criados")
print("\n Q007_label:")
print(df['Q007_label'].value_counts(dropna=False))
print("\n Q008_label:")
print(df['Q008_label'].value_counts(dropna=False))

 Labels descritivos criados

 Q007_label:
Q007_label
Não trabalho                 9248
Trabalho eventualmente        417
Trabalho período integral     232
Trabalho meio período         103
Name: count, dtype: int64

 Q008_label:
Q008_label
Até 10 horas     6700
11 a 20 horas    2201
21 a 30 horas     662
31 a 40 horas     340
Nenhuma            97
Name: count, dtype: int64


### 5.3 Criar Variáveis Ordinais

In [48]:
# Codificação ordinal Q007 (0 = não trabalha, 3 = período integral)
Q007_ORDINAL = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
df['Q007_ord'] = df['Q007'].map(Q007_ORDINAL)

# Codificação ordinal Q008 (0 = nenhuma, 5 = mais de 40h)
Q008_ORDINAL = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5}
df['Q008_ord'] = df['Q008'].map(Q008_ORDINAL)

print(" Variáveis ordinais criadas")
print("\n Distribuição Q007_ord:")
print(df['Q007_ord'].value_counts(dropna=False).sort_index())
print("\n Distribuição Q008_ord:")
print(df['Q008_ord'].value_counts(dropna=False).sort_index())

 Variáveis ordinais criadas

 Distribuição Q007_ord:
Q007_ord
0    9248
1     417
2     103
3     232
Name: count, dtype: int64

 Distribuição Q008_ord:
Q008_ord
0      97
1    6700
2    2201
3     662
4     340
Name: count, dtype: int64


### 5.4 Criar Variáveis Derivadas

In [49]:
# Variável binária: trabalha ou não
df['TRABALHA'] = (df['Q007'] != 'A').astype(int)

# Categorias simplificadas de trabalho
def categorizar_trabalho(row):
    if pd.isna(row['Q007']):
        return np.nan
    elif row['Q007'] == 'A':
        return 'Não trabalha'
    elif row['Q007'] in ['B', 'C']:
        return 'Trabalho parcial'
    else:  # D
        return 'Período integral'

df['CATEGORIA_TRABALHO'] = df.apply(categorizar_trabalho, axis=1)

# Carga horária numérica (ponto médio dos intervalos)
CARGA_HORARIA_NUM = {
    'A': 0,
    'B': 5,    # Até 10h -> média 5h
    'C': 15.5, # 11-20h -> média 15.5h
    'D': 25.5, # 21-30h -> média 25.5h
    'E': 35.5, # 31-40h -> média 35.5h
    'F': 45    # Mais de 40h -> estimativa 45h
}
df['CARGA_HORARIA_NUM'] = df['Q008'].map(CARGA_HORARIA_NUM)

print(" Variáveis derivadas criadas")
print("\n TRABALHA:")
print(df['TRABALHA'].value_counts())
print("\n CATEGORIA_TRABALHO:")
print(df['CATEGORIA_TRABALHO'].value_counts())
print("\n CARGA_HORARIA_NUM - Estatísticas:")
print(df['CARGA_HORARIA_NUM'].describe())

 Variáveis derivadas criadas

 TRABALHA:
TRABALHA
0    9248
1     752
Name: count, dtype: int64

 CATEGORIA_TRABALHO:
CATEGORIA_TRABALHO
Não trabalha        9248
Trabalho parcial     520
Período integral     232
Name: count, dtype: int64

 CARGA_HORARIA_NUM - Estatísticas:
count    10000.00
mean         9.66
std          7.89
min          0.00
25%          5.00
50%          5.00
75%         15.50
max         35.50
Name: CARGA_HORARIA_NUM, dtype: float64


---

## 6⃣ Validação de Consistência

Verificar se há inconsistências entre Q007 e Q008:

In [50]:
# Criar tabela cruzada
print(" Tabela Cruzada Q007 × Q008:")
tabela_cruzada = pd.crosstab(
    df['Q007_label'], 
    df['Q008_label'], 
    margins=True,
    dropna=False
)
print(tabela_cruzada)

# Identificar inconsistências
# Exemplo: Q007='A' (não trabalha) mas Q008 != 'A' (tem carga horária)
inconsistencias = df[(df['Q007'] == 'A') & (df['Q008'] != 'A') & df['Q008'].notna()]
print(f"\n Inconsistências encontradas: {len(inconsistencias):,}")
print(f"   ({len(inconsistencias)/len(df)*100:.2f}% do total)")

if len(inconsistencias) > 0:
    print("\nExemplos de inconsistências:")
    print(inconsistencias[['Q007', 'Q007_label', 'Q008', 'Q008_label']].head(10))

 Tabela Cruzada Q007 × Q008:
Q008_label                 11 a 20 horas  21 a 30 horas  31 a 40 horas  \
Q007_label                                                               
Não trabalho                        2035            486            145   
Trabalho eventualmente               112             99             78   
Trabalho meio período                 20             28             24   
Trabalho período integral             34             49             93   
All                                 2201            662            340   

Q008_label                 Até 10 horas  Nenhuma    All  
Q007_label                                               
Não trabalho                       6488       94   9248  
Trabalho eventualmente              125        3    417  
Trabalho meio período                31        0    103  
Trabalho período integral            56        0    232  
All                                6700       97  10000  

 Inconsistências encontradas: 9,154
   (91.54

In [51]:
# Tratar inconsistências (opcional)
# Se pessoa diz não trabalhar (Q007=A) mas tem carga horária, corrigir
print(" Corrigindo inconsistências...")

# Criar cópia das variáveis originais
df['Q007_original'] = df['Q007']
df['Q008_original'] = df['Q008']

# Regra: Se Q007=A mas Q008 tem valor diferente de A, 
# assumir que Q008 está correto e ajustar Q007
mask_inconsistente = (df['Q007'] == 'A') & (df['Q008'].notna()) & (df['Q008'] != 'A')
if mask_inconsistente.sum() > 0:
    # Mapear Q008 para Q007 aproximado
    def q008_para_q007(q008):
        if q008 in ['B', 'C']:  # Até 20h
            return 'C'  # Meio período
        elif q008 in ['D', 'E', 'F']:  # Mais de 20h
            return 'D'  # Período integral
        return 'A'
    
    df.loc[mask_inconsistente, 'Q007'] = df.loc[mask_inconsistente, 'Q008'].apply(q008_para_q007)
    print(f"   Corrigidos: {mask_inconsistente.sum():,} registros")
    
    # Atualizar variáveis derivadas
    df.loc[mask_inconsistente, 'Q007_label'] = df.loc[mask_inconsistente, 'Q007'].map(Q007_LABELS)
    df.loc[mask_inconsistente, 'Q007_ord'] = df.loc[mask_inconsistente, 'Q007'].map(Q007_ORDINAL)
    df.loc[mask_inconsistente, 'TRABALHA'] = 1
    df.loc[mask_inconsistente, 'CATEGORIA_TRABALHO'] = df.loc[mask_inconsistente].apply(categorizar_trabalho, axis=1)

print("\n Inconsistências tratadas")

 Corrigindo inconsistências...
   Corrigidos: 9,154 registros

 Inconsistências tratadas

 Inconsistências tratadas


---

## 7⃣ Estatísticas Finais

In [52]:
print(" ESTATÍSTICAS FINAIS DO DATASET")
print("=" * 60)

print(f"\n Tamanho do dataset:")
print(f"  Total de registros: {len(df):,}")
print(f"  Total de colunas: {len(df.columns)}")

print(f"\n Situação de trabalho (Q007):")
print(df['Q007_label'].value_counts())
print(f"\n  Percentuais:")
print(df['Q007_label'].value_counts(normalize=True) * 100)

print(f"\n Carga horária (Q008):")
print(df['Q008_label'].value_counts())
print(f"\n  Percentuais:")
print(df['Q008_label'].value_counts(normalize=True) * 100)

print(f"\n Trabalha ou não:")
print(df['TRABALHA'].value_counts())
print(f"\n  Percentual que trabalha: {df['TRABALHA'].mean() * 100:.2f}%")

print(f"\n Categoria de trabalho:")
print(df['CATEGORIA_TRABALHO'].value_counts())

 ESTATÍSTICAS FINAIS DO DATASET

 Tamanho do dataset:
  Total de registros: 10,000
  Total de colunas: 22

 Situação de trabalho (Q007):
Q007_label
Trabalho meio período        8626
Trabalho período integral     863
Trabalho eventualmente        417
Não trabalho                   94
Name: count, dtype: int64

  Percentuais:
Q007_label
Trabalho meio período        86.26
Trabalho período integral     8.63
Trabalho eventualmente        4.17
Não trabalho                  0.94
Name: proportion, dtype: float64

 Carga horária (Q008):
Q008_label
Até 10 horas     6700
11 a 20 horas    2201
21 a 30 horas     662
31 a 40 horas     340
Nenhuma            97
Name: count, dtype: int64

  Percentuais:
Q008_label
Até 10 horas     67.00
11 a 20 horas    22.01
21 a 30 horas     6.62
31 a 40 horas     3.40
Nenhuma           0.97
Name: proportion, dtype: float64

 Trabalha ou não:
TRABALHA
1    9906
0      94
Name: count, dtype: int64

  Percentual que trabalha: 99.06%

 Categoria de trabalho:
CATEGORIA_

In [53]:
# Resumo das novas colunas criadas
print("\n COLUNAS CRIADAS NESTE NOTEBOOK:")
print("=" * 60)
novas_colunas = [
    'Q007', 'Q008',  # Originais
    'Q007_label', 'Q008_label',  # Labels
    'Q007_ord', 'Q008_ord',  # Ordinais
    'TRABALHA',  # Binária
    'CATEGORIA_TRABALHO',  # Categórica simplificada
    'CARGA_HORARIA_NUM',  # Numérica
    'Q007_original', 'Q008_original'  # Backup
]
for col in novas_colunas:
    if col in df.columns:
        print(f"   {col}")


 COLUNAS CRIADAS NESTE NOTEBOOK:
   Q007
   Q008
   Q007_label
   Q008_label
   Q007_ord
   Q008_ord
   TRABALHA
   CATEGORIA_TRABALHO
   CARGA_HORARIA_NUM
   Q007_original
   Q008_original


---

## 8⃣ Salvar Dataset Processado

In [54]:
# Garantir que o diretório existe
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Salvar em formato parquet (mais eficiente)
print(f" Salvando dataset processado...")
df.to_parquet(OUTPUT_FILE, index=False, compression='snappy')

# Verificar tamanho do arquivo
tamanho_mb = OUTPUT_FILE.stat().st_size / (1024 * 1024)
print(f"\n Dataset salvo com sucesso!")
print(f"   Arquivo: {OUTPUT_FILE}")
print(f"   Tamanho: {tamanho_mb:.2f} MB")
print(f"   Registros: {len(df):,}")
print(f"   Colunas: {len(df.columns)}")

 Salvando dataset processado...

 Dataset salvo com sucesso!
   Arquivo: /home/interas/faculdade/ciencia-dados/enem-data-exploration/data/processed/enem_2023_trabalho_estudantil.parquet
   Tamanho: 0.26 MB
   Registros: 10,000
   Colunas: 22

 Dataset salvo com sucesso!
   Arquivo: /home/interas/faculdade/ciencia-dados/enem-data-exploration/data/processed/enem_2023_trabalho_estudantil.parquet
   Tamanho: 0.26 MB
   Registros: 10,000
   Colunas: 22


In [55]:
# Também salvar uma versão CSV para facilitar inspeção
CSV_FILE = PROCESSED_DIR / 'enem_2023_trabalho_estudantil_sample.csv'

# Salvar amostra (ou dataset completo se menor que 10.000)
n_sample = min(10000, len(df))
if n_sample < len(df):
    df_sample = df.sample(n=n_sample, random_state=42)
else:
    df_sample = df.copy()

df_sample.to_csv(CSV_FILE, index=False)

print(f"\nAmostra CSV salva: {CSV_FILE}")
print(f"   Registros: {len(df_sample):,}")


Amostra CSV salva: /home/interas/faculdade/ciencia-dados/enem-data-exploration/data/processed/enem_2023_trabalho_estudantil_sample.csv
   Registros: 10,000


---

## 9⃣ Sumário de Qualidade dos Dados

In [56]:
# Criar relatório de qualidade
print(" RELATÓRIO DE QUALIDADE DOS DADOS")
print("=" * 60)

colunas_trabalho = ['Q007', 'Q008', 'Q007_ord', 'Q008_ord', 'TRABALHA', 
                    'CATEGORIA_TRABALHO', 'CARGA_HORARIA_NUM']

for col in colunas_trabalho:
    if col in df.columns:
        total = len(df)
        nulos = df[col].isnull().sum()
        validos = total - nulos
        taxa_preenchimento = (validos / total) * 100
        
        print(f"\n{col}:")
        print(f"  Total: {total:,}")
        print(f"  Válidos: {validos:,} ({taxa_preenchimento:.2f}%)")
        print(f"  Nulos: {nulos:,} ({(nulos/total)*100:.2f}%)")
        
        if df[col].dtype in ['int64', 'float64']:
            print(f"  Média: {df[col].mean():.2f}")
            print(f"  Mediana: {df[col].median():.2f}")

 RELATÓRIO DE QUALIDADE DOS DADOS

Q007:
  Total: 10,000
  Válidos: 10,000 (100.00%)
  Nulos: 0 (0.00%)

Q008:
  Total: 10,000
  Válidos: 10,000 (100.00%)
  Nulos: 0 (0.00%)

Q007_ord:
  Total: 10,000
  Válidos: 10,000 (100.00%)
  Nulos: 0 (0.00%)
  Média: 2.03
  Mediana: 2.00

Q008_ord:
  Total: 10,000
  Válidos: 10,000 (100.00%)
  Nulos: 0 (0.00%)
  Média: 1.44
  Mediana: 1.00

TRABALHA:
  Total: 10,000
  Válidos: 10,000 (100.00%)
  Nulos: 0 (0.00%)
  Média: 0.99
  Mediana: 1.00

CATEGORIA_TRABALHO:
  Total: 10,000
  Válidos: 10,000 (100.00%)
  Nulos: 0 (0.00%)

CARGA_HORARIA_NUM:
  Total: 10,000
  Válidos: 10,000 (100.00%)
  Nulos: 0 (0.00%)
  Média: 9.66
  Mediana: 5.00


---

##  Checklist de Conclusão

- [x] Dataset original carregado
- [x] Variáveis Q007 e Q008 extraídas
- [x] Merge realizado com sucesso
- [x] Labels descritivos criados
- [x] Variáveis ordinais criadas
- [x] Variáveis derivadas criadas (TRABALHA, CATEGORIA_TRABALHO, CARGA_HORARIA_NUM)
- [x] Inconsistências identificadas e tratadas
- [x] Dataset salvo em formato Parquet
- [x] Amostra CSV gerada para inspeção
- [x] Relatório de qualidade gerado

---

##  Próximos Passos

 **`03_analise_descritiva_trabalho.ipynb`**

No próximo notebook, iremos:
1. Analisar a distribuição de estudantes por situação de trabalho
2. Calcular estatísticas descritivas de desempenho por grupo
3. Criar visualizações comparativas
4. Traçar o perfil socioeconômico dos estudantes que trabalham

---

*Notebook criado em: 10 de dezembro de 2025*  
*Dataset processado: enem_2023_trabalho_estudantil.parquet*